# 🀄 麻将YOLOv8训练 - 一键运行
点击菜单栏 **运行** → **运行所有单元格** ，然后等30-60分钟即可。

In [ ]:
# 第1步: 下载训练包
import os
os.chdir('/root')
if not os.path.exists('mahjong_train'):
    print('下载训练包...')
    os.system('git clone https://github.com/juhua458/mj-app.git mj_train_tmp')
    os.system('cp -r mj_train_tmp/yolo/train_package mahjong_train')
    os.system('rm -rf mj_train_tmp')
    print('下载完成!')
else:
    print('训练包已存在')

In [ ]:
# 第2步: 解压数据集
import os
os.chdir('/root/mahjong_train')
if not os.path.exists('images_raw'):
    print('解压图片数据集...')
    os.system('unzip -o images_raw.zip -d images_raw')
    print('解压完成!')
else:
    print('数据集已解压')

In [ ]:
# 第3步: 安装依赖
print('安装ultralytics...')
os.system('pip install ultralytics -q -i https://pypi.tuna.tsinghua.edu.cn/simple')
print('安装完成!')

In [ ]:
# 第4步: 准备数据集(YOLO格式)
print('准备数据集...')
os.chdir('/root/mahjong_train')
exec(open('prepare_data.py').read())

In [ ]:
# 第5步: 开始训练 (约30-60分钟)
print('开始训练，请耐心等待...')
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data='/root/mahjong_train/dataset/data.yaml',
    epochs=150,
    imgsz=640,
    batch=64,
    device=0,
    workers=8,
    project='/root/mahjong_train/runs',
    name='mahjong_detect',
    patience=30,
    save_period=10,
)
print('训练完成!')

In [ ]:
# 第6步: 导出TFLite模型
import shutil
print('导出TFLite模型...')
best_model = YOLO('/root/mahjong_train/runs/mahjong_detect/weights/best.pt')
best_model.export(format='tflite', imgsz=640)

# 复制到方便下载的位置
for root, dirs, files in os.walk('/root/mahjong_train/runs'):
    for f in files:
        if f.endswith('.tflite'):
            src = os.path.join(root, f)
            shutil.copy2(src, '/root/mahjong_train/mahjong_best.tflite')
            size_mb = os.path.getsize('/root/mahjong_train/mahjong_best.tflite') / 1024 / 1024
            print(f'TFLite模型: /root/mahjong_train/mahjong_best.tflite ({size_mb:.1f} MB)')
            break

pt_src = '/root/mahjong_train/runs/mahjong_detect/weights/best.pt'
if os.path.exists(pt_src):
    shutil.copy2(pt_src, '/root/mahjong_train/mahjong_best.pt')
    size_mb = os.path.getsize('/root/mahjong_train/mahjong_best.pt') / 1024 / 1024
    print(f'PT模型: /root/mahjong_train/mahjong_best.pt ({size_mb:.1f} MB)')

print('\n🎉 全部完成! 从左侧文件管理器下载 mahjong_best.tflite')